# Naïve Bayes Classifier Implementation

This notebook implements a Naïve Bayes classifier for spam detection in two parts:
- **Part 1**: Manual implementation from scratch (no scikit-learn)
- **Part 2**: Using scikit-learn's MultinomialNB

In [43]:
# Import libraries
import re
import math
from collections import defaultdict, Counter

# Dataset
documents = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

# Test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

print("Dataset loaded successfully!")
print(f"Total documents: {len(documents)}")

Dataset loaded successfully!
Total documents: 11


## Part 1: Manual Naïve Bayes Implementation

In [ ]:
def preprocess(text):
    # Lowercase, remove punctuation, tokenize
    return re.findall(r"\b\w+\b", text.lower())

def build_bow(docs):
    # Build vocabulary and word counts per class
    class_word_counts = defaultdict(Counter)
    class_doc_counts = defaultdict(int)
    total_docs = 0
    
    for text, label in docs:
        total_docs += 1
        class_doc_counts[label] += 1
        tokens = preprocess(text)
        class_word_counts[label].update(tokens)
    
    vocab = set()
    for counts in class_word_counts.values():
        vocab.update(counts.keys())
    
    return vocab, class_word_counts, class_doc_counts, total_docs

def compute_priors(class_doc_counts, total_docs):
    # Calculate P(class)
    return {label: count / total_docs for label, count in class_doc_counts.items()}

def compute_likelihoods(class_word_counts, vocab):
    # Calculate P(word|class) with Laplace smoothing
    V = len(vocab)
    likelihoods = {}
    total_words_per_class = {}
    
    for label, counts in class_word_counts.items():
        total = sum(counts.values())
        total_words_per_class[label] = total
        likelihoods[label] = {}
        for word in vocab:
            likelihoods[label][word] = (counts.get(word, 0) + 1) / (total + V)
    
    return likelihoods, total_words_per_class, V

def predict_naive_bayes(sentence, priors, likelihoods, total_words_per_class, V, vocab):
    # Predict class using log probabilities
    tokens = preprocess(sentence)
    log_probs = {}
    
    for label in priors:
        log_prob = math.log(priors[label])
        for word in tokens:
            if word in vocab:
                prob = likelihoods[label][word]
            else:
                prob = 1 / (total_words_per_class[label] + V)
            log_prob += math.log(prob)
        log_probs[label] = log_prob
    
    predicted = max(log_probs, key=log_probs.get)
    return predicted, log_probs

print("Functions defined!")

Functions defined!


### Step 2: Train and Display Results

In [45]:
# Build model
vocab, class_word_counts, class_doc_counts, total_docs = build_bow(documents)
priors = compute_priors(class_doc_counts, total_docs)
likelihoods, total_words_per_class, V = compute_likelihoods(class_word_counts, vocab)

print("PART 1 — Manual Naïve Bayes")
print("=" * 50)
print(f"\nVocabulary: {sorted(vocab)}")
print(f"\nWord counts per class:")
for label in sorted(class_word_counts.keys()):
    print(f"{label}: {dict(class_word_counts[label])}")
print(f"\nPrior probabilities:")
for label, p in sorted(priors.items()):
    print(f"P({label}) = {p:.4f}")

PART 1 — Manual Naïve Bayes

Vocabulary: ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

Word counts per class:
HAM: {'hi': 1, 'mom': 1, 'how': 1, 'are': 2, 'you': 2, 'we': 1, 'still': 1, 'on': 1, 'for': 1, 'dinner': 1, 'let': 1, 's': 1, 'catch': 1, 'up': 1, 'tomorrow': 2, 'at': 2, 'the': 3, 'office': 2, 'meeting': 2, '3': 1, 'pm': 1, 'team': 1, 'in': 1, 'can': 1, 'send': 1, 'report': 1}
SPAM: {'free': 2, 'money': 1, 'now': 1, 'lowest': 1, 'price': 1, 'for': 2, 'your': 1, 'meds': 1, 'win': 1, 'a': 1, 'iphone': 1, 'today': 1, 'get': 1, '50': 1, 'off': 1, 'limited': 1, 'time': 1, 'click': 1, 'here': 1, 'prizes': 1}

Prior probabilities:
P(HAM) = 0.5455
P(SPAM) = 0.4545


### Step 3: Predictions

In [46]:
print("\nPredictions:")
for sentence in test_sentences:
    predicted, log_probs = predict_naive_bayes(
        sentence, priors, likelihoods, total_words_per_class, V, vocab
    )
    print(f'\n"{sentence}" → {predicted}')


Predictions:

"Limited offer, click here!" → SPAM

"Meeting at 2 PM with the manager." → HAM


## Part 2: Using Scikit-Learn

In [47]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [48]:
# Prepare and train
texts = [text for text, _ in documents]
labels = [label for _, label in documents]

vectorizer = CountVectorizer(lowercase=True, token_pattern=r"\b\w+\b")
X_train = vectorizer.fit_transform(texts)

clf = MultinomialNB()
clf.fit(X_train, labels)

print("PART 2 — Scikit-Learn")
print("=" * 50)
print(f"Vocabulary: {sorted(vectorizer.vocabulary_.keys())}")

PART 2 — Scikit-Learn
Vocabulary: ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']


In [49]:
# Predictions
X_test = vectorizer.transform(test_sentences)
predictions = clf.predict(X_test)

print("\nPredictions:")
for sentence, pred in zip(test_sentences, predictions):
    print(f'"{sentence}" → {pred}')


Predictions:
"Limited offer, click here!" → SPAM
"Meeting at 2 PM with the manager." → HAM
